In [1]:
import torch

torch.__version__

'2.12.0+cu126'

In [2]:
from tensorflow.keras.datasets import imdb

max_features = 10000

(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words=max_features
)

In [3]:
print(type(X_train))
print(len(X_train))

print(X_train[0])
print(y_train[0])

<class 'numpy.ndarray'>
25000
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
1


In [4]:
## decode reviews
word_index = imdb.get_word_index()

reverse_word_index = {
    value: key
    for key, value in word_index.items()
}

In [5]:
def decode_review(review):

    return " ".join(
        reverse_word_index.get(i - 3, "?")
        for i in review
    )

In [6]:
print(decode_review(X_train[0]))

? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't you thi

In [7]:
max_len=500

In [8]:
def preprocess_review(review):

    review = torch.tensor(
        review,
        dtype=torch.long
    )

    # pre-truncation

    review = review[-max_len:]

    # pre-padding

    if len(review) < max_len:

        padding = torch.zeros(
            max_len - len(review),
            dtype=torch.long
        )

        review = torch.cat(
            [padding, review]
        )

    return review

In [9]:
## apply to train set
X_train = torch.stack([
    preprocess_review(review)
    for review in X_train
])

## apply to test set
X_test = torch.stack([
    preprocess_review(review)
    for review in X_test
])

In [10]:
print(X_train.shape)

torch.Size([25000, 500])


In [11]:
X_train[0]

tensor([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,   

In [12]:
## Step 5 --> Convert labels
y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [13]:
print(y_train.shape)

torch.Size([25000])


In [14]:
## train and test dataset
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(
    X_train,
    y_train
)

test_dataset = TensorDataset(
    X_test,
    y_test
)

In [15]:
from torch.utils.data import DataLoader

## create data loader

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size
)

In [16]:
import torch.nn as nn
class SimpleRNNClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=10000,
            embedding_dim=128
        )

        self.rnn = nn.RNN(
            input_size=128,
            hidden_size=128,
            batch_first=True,
            nonlinearity='relu'
        )

        self.fc = nn.Linear(
            128,
            1
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        embedded = self.embedding(x)

        output, hidden = self.rnn(
            embedded
        )

        hidden = hidden.squeeze(0)

        output = self.fc(hidden)

        output = self.sigmoid(output)

        return output

In [17]:
## create the model
model = SimpleRNNClassifier()

In [18]:
## loss function
criterion = nn.BCELoss()

In [19]:
## optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [20]:
print(torch.cuda.is_available())

True


In [21]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [22]:
model.to(device)

SimpleRNNClassifier(
  (embedding): Embedding(10000, 128)
  (rnn): RNN(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [23]:
print(next(model.parameters()).device)

cuda:0


In [24]:
print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 3050 Laptop GPU


In [25]:
print(model)

SimpleRNNClassifier(
  (embedding): Embedding(10000, 128)
  (rnn): RNN(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [26]:
X_batch, y_batch = next(iter(train_loader))

print(X_batch.shape)
print(y_batch.shape)

torch.Size([32, 500])
torch.Size([32])


In [27]:
## now we'll move the batch to gpu
X_batch = X_batch.to(device)

In [28]:
## embedding layer

embedded = model.embedding(X_batch)

print(embedded.shape)

torch.Size([32, 500, 128])


In [29]:
output, hidden = model.rnn(embedded)

print(output.shape)
print(hidden.shape)

torch.Size([32, 500, 128])
torch.Size([1, 32, 128])


In [30]:
hidden = hidden.squeeze(0)

print(hidden.shape)

torch.Size([32, 128])


In [31]:
## linear layer
logits = model.fc(hidden)

print(logits.shape)

torch.Size([32, 1])


In [32]:
## sigmoid
preds = model.sigmoid(logits)

print(preds.shape)
print(preds[:5])

torch.Size([32, 1])
tensor([[0.3912],
        [0.4285],
        [0.3753],
        [0.3931],
        [0.4585]], device='cuda:0', grad_fn=<SliceBackward0>)


### Training loop

In [33]:
X_batch, y_batch = next(iter(train_loader))

X_batch = X_batch.to(device)
y_batch = y_batch.to(device)

In [34]:
predictions = model(X_batch).squeeze()

In [35]:
print(predictions.shape)
print(y_batch.shape)

torch.Size([32])
torch.Size([32])


In [36]:
loss = criterion(
    predictions,
    y_batch
)
print(loss)

tensor(0.6967, device='cuda:0', grad_fn=<BinaryCrossEntropyBackward0>)


In [37]:
## clear old gradient

optimizer.zero_grad()

In [38]:
## backpropogation

loss.backward()

In [39]:
## update weights
optimizer.step()

In [40]:
X_batch, y_batch = next(iter(train_loader))

X_batch = X_batch.to(device)
y_batch = y_batch.to(device)

optimizer.zero_grad()

predictions = model(X_batch).squeeze()

loss = criterion(
    predictions,
    y_batch
)

print(loss.item())

loss.backward()

optimizer.step()

0.6851726770401001


In [41]:
## full training loop

epochs = 10

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        predictions = model(X_batch).squeeze()

        loss = criterion(
            predictions,
            y_batch
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs}, "
        f"Loss: {running_loss/len(train_loader):.4f}"
    )

Epoch 1/10, Loss: 0.7417
Epoch 2/10, Loss: 0.5206
Epoch 3/10, Loss: 0.4402
Epoch 4/10, Loss: 0.3638
Epoch 5/10, Loss: 0.3391
Epoch 6/10, Loss: 0.3472
Epoch 7/10, Loss: 0.3269
Epoch 8/10, Loss: 0.5557
Epoch 9/10, Loss: 0.2393
Epoch 10/10, Loss: 0.2331


In [42]:
## evaluate on a test set
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        predictions = model(X_batch).squeeze()

        predictions = (
            predictions >= 0.5
        ).float()

        correct += (
            predictions == y_batch
        ).sum().item()

        total += y_batch.size(0)

accuracy = correct / total

print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.7850


In [43]:
torch.save(
    model.state_dict(),
    "model-files-2/simple_rnn_imdb.pth"
)